In [1]:
import os 
os.chdir('../')
%pwd

'c:\\Users\\DELL\\Desktop\\drink_quality_prediction'

In [2]:
#entity 

from dataclasses import dataclass 
from pathlib import Path



@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transformer_path: Path
    X_train_path: Path
    X_test_path: Path
    y_train_path: Path
    y_test_path: Path


In [3]:
from mlproject import logger
from mlproject.utils.common import read_yaml, create_directories

from  mlproject.constants import * 




In [4]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories



class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            transformer_path=config.transformer_path,
            X_train_path=config.X_train_path,
            X_test_path=config.X_test_path,
            y_train_path=config.y_train_path,
            y_test_path=config.y_test_path,
        )

        return data_transformation_config

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from mlproject import logger


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def initiate_data_transformation(self):

        try:
            logger.info("Starting Data Transformation")

            # ===============================
            # 1️⃣ Load Data
            # ===============================
            logger.info("Loading dataset")
            df = pd.read_csv(self.config.data_path)
            logger.info(f"Dataset loaded with shape: {df.shape}")

            # Remove duplicates
            df.drop_duplicates(inplace=True)
            logger.info(f"Shape after removing duplicates: {df.shape}")

            # ===============================
            # 2️⃣ Create Binary Target
            # ===============================
            logger.info("Creating binary target column")
            df["quality_binary"] = (df["quality"] >= 6).astype(int)

            # ===============================
            # 3️⃣ Define Features & Target
            # ===============================
            X = df.drop(columns=["quality", "quality_binary"])
            y = df["quality_binary"]

            # ===============================
            # 4️⃣ Train-Test Split
            # ===============================
            logger.info("Splitting data into train and test")
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.30,
                random_state=42,
                stratify=y
            )

            logger.info(f"X_train shape: {X_train.shape}")
            logger.info(f"X_test shape: {X_test.shape}")

            # ===============================
            # 5️⃣ Log Transform Skewed Columns
            # ===============================
            logger.info("Applying log transformation to skewed columns")

            skewed_cols = [
                'residual sugar',
                'chlorides',
                'free sulfur dioxide',
                'total sulfur dioxide',
                'sulphates'
            ]

            X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
            X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

            # ===============================
            # 6️⃣ Feature Scaling
            # ===============================
            logger.info("Applying Standard Scaling")

            scaler = StandardScaler()

            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Convert back to DataFrame
            X_train_scaled = pd.DataFrame(
                X_train_scaled,
                columns=X_train.columns
            )

            X_test_scaled = pd.DataFrame(
                X_test_scaled,
                columns=X_test.columns
            )

            y_train = pd.DataFrame(y_train, columns=["quality_binary"])
            y_test = pd.DataFrame(y_test, columns=["quality_binary"])

            # ===============================
            # 7️⃣ Save Artifacts as CSV
            # ===============================
            logger.info("Saving transformation artifacts as CSV")

            os.makedirs(self.config.root_dir, exist_ok=True)

            X_train_scaled.to_csv(self.config.X_train_path, index=False)
            X_test_scaled.to_csv(self.config.X_test_path, index=False)
            y_train.to_csv(self.config.y_train_path, index=False)
            y_test.to_csv(self.config.y_test_path, index=False)

            logger.info("Train-test CSV files saved successfully")

            # Save scaler
            joblib.dump(scaler, self.config.transformer_path)
            logger.info("Scaler saved successfully")

            logger.info("Data Transformation Completed Successfully")

            return (
                self.config.X_train_path,
                self.config.X_test_path,
                self.config.y_train_path,
                self.config.y_test_path
            )

        except Exception as e:
            logger.exception("Error occurred during Data Transformation")
            raise e

In [6]:
try:
    config = ConfigurationManager()
    
    data_transformation_config = config.get_data_transformation_config()
    
    data_transformation = DataTransformation(config=data_transformation_config)
    
    X_train_path, X_test_path, y_train_path, y_test_path = \
        data_transformation.initiate_data_transformation()
    
    print("Data Transformation Completed Successfully!")
    print("Saved Files:")
    print("X_train:", X_train_path)
    print("X_test :", X_test_path)
    print("y_train:", y_train_path)
    print("y_test :", y_test_path)

except Exception as e:
    raise e

[2026-03-16 01:51:13,430] INFO: common: yaml file: config\config.yaml loaded successfully
[2026-03-16 01:51:13,432] INFO: common: yaml file: params.yaml loaded successfully
[2026-03-16 01:51:13,433] INFO: common: yaml file: schema.yaml loaded successfully
[2026-03-16 01:51:13,434] INFO: common: created directory at: artifacts
[2026-03-16 01:51:13,435] INFO: common: created directory at: artifacts/data_transformation
[2026-03-16 01:51:13,436] INFO: 3042348043: Starting Data Transformation
[2026-03-16 01:51:13,437] INFO: 3042348043: Loading dataset
[2026-03-16 01:51:13,441] INFO: 3042348043: Dataset loaded with shape: (1599, 12)
[2026-03-16 01:51:13,443] INFO: 3042348043: Shape after removing duplicates: (1359, 12)
[2026-03-16 01:51:13,444] INFO: 3042348043: Creating binary target column
[2026-03-16 01:51:13,446] INFO: 3042348043: Splitting data into train and test
[2026-03-16 01:51:13,449] INFO: 3042348043: X_train shape: (951, 11)
[2026-03-16 01:51:13,450] INFO: 3042348043: X_test shap

In [8]:
import pandas as pd

X_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)

In [9]:
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,-0.233530,1.031057,-1.332609,-0.585043,-0.245621,-0.919095,-0.205345,-0.008530,0.513858,-0.731819,-0.792313
1,-0.580000,2.933478,-0.925979,-0.852889,1.881745,-0.400647,1.257934,-0.263521,-0.077286,-0.531765,-1.341681
2,-0.753234,0.840815,-1.383437,-0.585043,-0.442918,1.741297,0.760705,0.639574,1.893194,0.943258,-0.517629
3,-1.561663,0.052669,-1.180123,-0.852889,-0.842556,-1.387446,-1.455905,-1.325986,2.681385,-0.465929,0.855793
4,-0.637744,-0.001686,-0.671836,-0.335067,-0.509054,0.209360,-0.344352,-1.841281,0.251127,-0.270893,1.496723
...,...,...,...,...,...,...,...,...,...,...,...
946,-0.810979,-0.110396,-0.214378,-0.457971,-0.333104,-0.008031,-0.731870,-1.230364,0.579541,1.113713,0.672670
947,-0.522255,1.139767,-1.027637,-0.852889,-0.028327,0.209360,0.371204,-0.890376,-1.259574,-1.143892,-0.426067
948,1.210093,0.107024,-0.214378,0.328114,0.079570,0.489135,0.371204,1.436422,-0.602747,-0.079457,-0.151383
949,-0.868724,0.161379,-1.230951,-0.852889,-0.071625,0.489135,-0.078442,0.055218,0.842271,-0.142877,-0.426067






i think it is not correct
def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status
if any col. is missing but last col is in allschema so it will give true

You’re absolutely right — your current logic is incorrect. 👍

The problem is this:

You are updating validation_status inside the loop

So even if one column fails, a later valid column will overwrite it back to True

Therefore, the result depends on the last column checked

🔍 Why Your Current Code Fails
Example:

CSV columns:      ["A", "B", "C"]
Schema columns:   ["A", "B"]
Loop execution:

A → valid → status = True

B → valid → status = True

C → invalid → status = False

✅ Works in this case.

But if order changes:

CSV columns: ["A", "C", "B"]
A → True

C → False

B → True ← overwrites False

❌ Final result = True (WRONG)

✅ Correct Approach
You should:

Assume validation is True

If any column fails, set it to False

Break the loop immediately (optional but cleaner)

Write to file only once

✔️ Clean & Correct Version
def validate_all_columns(self) -> bool:
    try:
        data = pd.read_csv(self.config.unzip_data_dir)
        all_cols = list(data.columns)
        all_schema = list(self.config.all_schema.keys())

        # Assume valid initially
        validation_status = True

        for col in all_cols:
            if col not in all_schema:
                validation_status = False
                break  # No need to check further

        # Write status once
        with open(self.config.STATUS_FILE, 'w') as f:
            f.write(f"Validation status: {validation_status}")

        return validation_status

    except Exception as e:
        print(e)
        return False
🚀 Even Better (Pythonic Way)
You don’t even need a loop:

def validate_all_columns(self) -> bool:
    try:
        data = pd.read_csv(self.config.unzip_data_dir)
        all_cols = set(data.columns)
        all_schema = set(self.config.all_schema.keys())

        validation_status = all_cols.issubset(all_schema)

        with open(self.config.STATUS_FILE, 'w') as f:
            f.write(f"Validation status: {validation_status}")

        return validation_status

    except Exception as e:
        print(e)
        return False
Why this is better:
Cleaner

Faster

No manual looping

No logical overwrite bug

🎯 If You Also Want to Check Missing Columns
If you want to ensure:

No extra columns

No missing columns

Use:

validation_status = all_cols == all_schema
That enforces an exact match.

If you'd like, I can also show you a version that prints which columns are missing or extra (very useful for debugging ML pipelines).


---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[11], line 8
      5     data_validation.validate_all_columns()
      7 except Exception as e:
----> 8     raise e

Cell In[11], line 5
      3     data_validation_config  = config.get_data_validation_config()
      4     data_validation = DataValidation(data_validation_config)
----> 5     data_validation.validate_all_columns()
      7 except Exception as e:
      8     raise e

Cell In[10], line 32
     28     return validation_status 
     31 except Exception as e :
---> 32     raise e

Cell In[10], line 24
     20         validation_status=False 
     21         break 
---> 24 with open(self.config.STATUS_FILE,'w') as f:
     25     f.write(f"Validation status: {validation_status}")
     28 return validation_status 
...
    321         "you can use builtins' open."
    322     )
--> 324 return io_open(file, *args, **kwargs)

FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/data_validation/status/txt'
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...





my codes
pwd
import os
os.chdir('..')
%pwd
import pandas as pd 
data  = pd.read_csv('artifacts\data_ingestion\Drinks-data.csv')
data.head()
data.info()
from dataclasses import dataclass
from pathlib import Path 

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path 
    STATUS_FILE : str 
    unzip_data_dir: Path 
    all_schema: dict
#config_Manger 
from mlproject.utils.common import read_yaml,create_directories

from mlproject.constants import *
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH 
    ):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        
        create_directories([self.config.artifacts_root])
        
        
    def  get_data_validation_config(self)->DataValidationConfig:
        config = self.config.data_validation 
        schema = self.schema.COLUMNS
        
        
        
        create_directories([config.root_dir]) 
        
        data_validation_config = DataValidationConfig(
                root_dir = config.root_dir,
                STATUS_FILE =config.STATUS_FILE,
                
                unzip_data_dir = config.unzip_data_dir,

                all_schema= schema
        )
        
        
        return data_validation_config
try: 
    config= ConfigurationManager()
    data_validation_config  = config.get_data_validation_config()
    data_validation = DataValidation(data_validation_config)
    data_validation.validate_all_columns()
    
except Exception as e:
    raise e

my teacher code
import os
%pwd
os.chdir("../")
%pwd
import pandas as pd
data = pd.read_csv("artifacts/data_ingestion/Drinks-data.csv")
data.head()
data.info()
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
        

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config
import os
from mlProject import logger
class DataValiadtion: 
    def __init__(self, config: DataValidationConfig):
        self.config = config


    def validate_all_columns(self)-> bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            all_schema = self.config.all_schema.keys()

            
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Validation status: {validation_status}")

            return validation_status
        
        except Exception as e:
            raise e


try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValiadtion(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

Good 👌 now your error is not about logic — it’s about path creation.

❌ The Real Problem
FileNotFoundError: 
'artifacts/data_validation/status/txt'
Python is trying to open:

artifacts/data_validation/status/txt
But that folder does not exist.

⚠️ open(..., 'w') does NOT create missing directories.
It only creates the file if the folder already exists.

🔎 Why Teacher Code Works
In your teacher’s YAML config, the folder structure probably exists already.

In your project, this folder is missing:

artifacts/
 └── data_validation/
      └── status/   ❌ (not created)
Your code only creates:

create_directories([config.root_dir])
That creates:

artifacts/data_validation/
But NOT:

artifacts/data_validation/status/
✅ Proper Fix (Best Practice)
Modify your validate_all_columns() method like this:

import os

def validate_all_columns(self) -> bool:
    try:
        data = pd.read_csv(self.config.unzip_data_dir)
        all_cols = set(data.columns)
        all_schema = set(self.config.all_schema.keys())

        validation_status = all_cols.issubset(all_schema)

        # ✅ Create directory for status file before writing
        os.makedirs(os.path.dirname(self.config.STATUS_FILE), exist_ok=True)

        with open(self.config.STATUS_FILE, 'w') as f:
            f.write(f"Validation status: {validation_status}")

        return validation_status

    except Exception as e:
        raise e
🔥 Cleaner Professional Way (Using Path)
Since you're already using Path, do this instead:

from pathlib import Path

def validate_all_columns(self) -> bool:
    try:
        data = pd.read_csv(self.config.unzip_data_dir)
        all_cols = set(data.columns)
        all_schema = set(self.config.all_schema.keys())

        validation_status = all_cols.issubset(all_schema)

        # ✅ Ensure parent directory exists
        status_path = Path(self.config.STATUS_FILE)
        status_path.parent.mkdir(parents=True, exist_ok=True)

        with open(status_path, 'w') as f:
            f.write(f"Validation status: {validation_status}")

        return validation_status

    except Exception as e:
        raise e
This is the cleanest solution.

🚨 Also Check This (Very Important)
Look at your error carefully:

artifacts/data_validation/status/txt
That looks wrong ❗

It should probably be:

artifacts/data_validation/status.txt
Notice the difference:

❌ status/txt → folder named status, file named txt
✅ status.txt → correct filename

🎯 Check Your YAML
Open your config.yaml and verify:

STATUS_FILE: artifacts/data_validation/status.txt
NOT:

STATUS_FILE: artifacts/data_validation/status/txt
If it's written like the second one → that’s the real issue.

💡 Final Checklist
✅ Fix YAML path (status.txt)

✅ Ensure directory creation before writing

✅ Use Path(...).parent.mkdir(parents=True, exist_ok=True)

If you want, send your config.yaml and I’ll point out the exact mistake instantly.


extract data transformation part 
print("hello")
import os
pwd
os.chdir('..')
pwd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

data_path = 'data\Drinks-data.csv'
df=pd.read_csv(data_path)
df
df.info()
df.describe()
df.isna().sum()
df.hist(bins = 50 , figsize=(20,15),color = 'orange')
from sklearn.model_selection import train_test_split
df.quality.value_counts()
print(df.duplicated().sum())
df.drop_duplicates(inplace=True)

X = df.drop(columns=['quality'],axis=1)
y=df.quality
X_train , X_test , y_train, y_test = train_test_split(X,y,test_size=.20,random_state=42)
y_test
print("dataframe_size : " , df.shape)
print("X_train :",X_train.shape)
print("X_test",X_test.shape)
print('y_train',y_train.shape)
print('y_test',y_test.shape)
X_train.select_dtypes(include=np.number)
num_features = X_train.select_dtypes(include=np.number).columns.to_list()

def  detect_outliers_iqr(df):
    outlier_summary ={} 
    
    for col in df.columns:
        Q1 = df[col].quantile(.25)
        Q3 = df[col].quantile(.75)
        IQR = Q3-Q1 
        
        lower_bond =  Q1 - 1.5 * IQR 
        upper_bound = Q3 + 1.5 * IQR
        
        
        outliers = df[ (df[col]<lower_bond)|( df[col]>upper_bound)]
        
        
        outlier_summary[col] = len(outliers)
    
    return pd.DataFrame.from_dict(outlier_summary, orient='index', columns=['Outlier Count'])

        
outlier_report = detect_outliers_iqr(df[num_features])

print(outlier_report.sort_values(by="Outlier Count", ascending=False))
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(20, 12))
X_train[num_features].boxplot(rot=90)
plt.title("Boxplot for Outlier Detection")
plt.show() 
plt.figure(figsize=(15, 25))

for i, col in enumerate(num_features, 1):
    plt.subplot(6, 2, i)
    sns.boxplot(x=X[col])
    plt.title(col)

plt.tight_layout()
plt.show()
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()



X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)
plt.figure(figsize=(15, 25))
plt.title("after scaling")

for i, col in enumerate(num_features, 1):
    plt.subplot(6, 2, i)
    sns.boxplot(x=X_train_scaled[col])
    plt.title(col)

plt.tight_layout()
plt.show()
skewed_cols = [
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'sulphates'
]

X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

from sklearn.linear_model import ElasticNet
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

# Train
pipeline.fit(X_train, y_train)

# Extract trained logistic regression model
log_reg = pipeline.named_steps["model"]
print(log_reg.coef_.shape)
print("""So in your case:
6 classes & 11 features""")

# If binary classification
if log_reg.coef_.shape[0] == 1:
    importance = pd.Series(
        log_reg.coef_[0],
        index=X_train.columns
    )

else:
    # Multiclass case → take mean absolute coefficient across classes
    importance = pd.Series(
        np.abs(log_reg.coef_).mean(axis=0),
        index=X_train.columns
    )
    
    

# Sort by importance
importance = importance.sort_values(key=abs, ascending=False)

print(importance)
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

importance = pd.Series(model.feature_importances_, index=X_train.columns)
importance = importance.sort_values(ascending=False)

print(importance)
y.unique()
df1 = df.copy()
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
# 1️⃣ Load Data
# ===============================

data_path = 'data\\Drinks-data.csv'
df = pd.read_csv(data_path)

print("Original Shape:", df.shape)
print(df.quality.value_counts())

df.drop_duplicates(inplace=True)
print("After removing duplicates:", df.shape)


# ===============================
# 2️⃣ Create Binary Target
# ===============================
# 0 = Bad (3,4,5)
# 1 = Good (6,7,8)

df["quality_binary"] = (df["quality"] >= 6).astype(int)

print("\nBinary Target Distribution:")
print(df["quality_binary"].value_counts())


# ===============================
# 3️⃣ Define X and y
# ===============================

X = df.drop(columns=["quality", "quality_binary"])
y = df["quality_binary"]


# ===============================
# 4️⃣ Train-Test Split (Stratified)
# ===============================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("\nTrain/Test Shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# ===============================
# 5️⃣ Log Transform Skewed Columns
# ===============================

skewed_cols = [
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'sulphates'
]

X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
X_test[skewed_cols] = np.log1p(X_test[skewed_cols])


# ===============================
# 6️⃣ Logistic Regression (Balanced)
# ===============================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, ConfusionMatrixDisplay

log_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, class_weight='balanced'))
])

log_pipeline.fit(X_train, y_train)

y_pred_log = log_pipeline.predict(X_test)

print("\n===== Logistic Regression (Binary) =====")
print("Accuracy :", accuracy_score(y_test, y_pred_log))
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall   :", recall_score(y_test, y_pred_log))
print("F1 Score :", f1_score(y_test, y_pred_log))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

ConfusionMatrixDisplay.from_estimator(
    log_pipeline,
    X_test,
    y_test,
    display_labels=["Bad", "Good"],
    cmap="Blues"
)
plt.title("Logistic Regression Confusion Matrix")
plt.show()


# ===============================
# 7️⃣ Random Forest (Balanced)
# ===============================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("\n===== Random Forest (Binary) =====")
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

ConfusionMatrixDisplay.from_estimator(
    rf_model,
    X_test,
    y_test,
    display_labels=["Bad", "Good"],
    cmap="Greens"
)
plt.title("Random Forest Confusion Matrix")
plt.show()
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd


def evaluate_models(X_train, X_test, y_train, y_test):
    
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, class_weight='balanced'))
        ]),
        
        "Random Forest": RandomForestClassifier(
            n_estimators=400,
            max_depth=12,
            class_weight='balanced',
            random_state=42
        ),
        
        "Gradient Boosting": GradientBoostingClassifier(),
        
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=400,
            random_state=42
        ),
        
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(class_weight='balanced'))
        ]),
        
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        
        "Decision Tree": DecisionTreeClassifier(
            class_weight='balanced',
            random_state=42
        ),
        
        "Naive Bayes": GaussianNB()
    }
    
    results = {}
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        results[name] = {
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred)
        }
    
    # Convert to DataFrame
    results_df = pd.DataFrame(results).T
    
    # Sort by F1 Score
    results_df = results_df.sort_values(by="F1 Score", ascending=False)
    
    print("\n===== Top 5 Models (Sorted by F1 Score) =====\n")
    print(results_df.head(5))
    
    return results_df
results_df = evaluate_models(X_train, X_test, y_train, y_test)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score
import pandas as pd

import mlflow

mlflow.set_experiment("MLflow Quickstart")


# Enable autologging for scikit-learn
mlflow.sklearn.autolog()

def fine_tune_top_models(X_train, X_test, y_train, y_test):
    
    # -----------------------------
    # Parameter Grids
    # -----------------------------
    
    rf_params = {
        "n_estimators": [300, 500, 800],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }
    
    et_params = {
        "n_estimators": [300, 500, 800],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }
    
    # -----------------------------
    # Models
    # -----------------------------
    
    rf = RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    )
    
    et = ExtraTreesClassifier(
        random_state=42
    )
    
    # -----------------------------
    # GridSearch
    # -----------------------------
    
    rf_grid = GridSearchCV(
        rf,
        rf_params,
        cv=5,
        scoring='f1',
        n_jobs=-1
    )
    
    et_grid = GridSearchCV(
        et,
        et_params,
        cv=5,
        scoring='f1',
        n_jobs=-1
    )
    
    print("Tuning Random Forest...")
    rf_grid.fit(X_train, y_train)
    
    print("Tuning Extra Trees...")
    et_grid.fit(X_train, y_train)
    
    # -----------------------------
    # Best Estimators
    # -----------------------------
    
    best_rf = rf_grid.best_estimator_
    best_et = et_grid.best_estimator_
    
    # -----------------------------
    # Evaluate on Test Data
    # -----------------------------
    
    models = {
        "Tuned Random Forest": best_rf,
        "Tuned Extra Trees": best_et
    }
    
    results = {}
    
    for name, model in models.items():
        y_pred = model.predict(X_test)
        
        results[name] = {
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred)
        }
        
        print(f"\n{name} Classification Report:")
        print(classification_report(y_test, y_pred))
    
    results_df = pd.DataFrame(results).T
    results_df = results_df.sort_values(by="F1 Score", ascending=False)
    
    print("\n===== Tuned Model Results =====\n")
    print(results_df)
    
    print("\n===== Best Parameters =====\n")
    print("Random Forest:", rf_grid.best_params_)
    print("Extra Trees:", et_grid.best_params_)
    
    return results_df, best_rf, best_et
results_df, best_rf_model, best_et_model = fine_tune_top_models(
    X_train, X_test, y_train, y_test
)
from sklearn.model_selection import RandomizedSearchCV
def quick_tune_models(X_train, X_test, y_train, y_test):

    with mlflow.start_run():

        rf_params = {
            "n_estimators": [100, 200],
            "max_depth": [None, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2]
        }

        et_params = {
            "n_estimators": [100, 200],
            "max_depth": [None, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2]
        }

        rf = RandomForestClassifier(class_weight='balanced', random_state=42)
        et = ExtraTreesClassifier(random_state=42)

        rf_search = RandomizedSearchCV(
            rf, rf_params,
            n_iter=6,
            cv=3,
            scoring='f1',
            n_jobs=-1,
            random_state=42
        )

        et_search = RandomizedSearchCV(
            et, et_params,
            n_iter=6,
            cv=3,
            scoring='f1',
            n_jobs=-1,
            random_state=42
        )

        rf_search.fit(X_train, y_train)
        et_search.fit(X_train, y_train)

        best_rf = rf_search.best_estimator_
        best_et = et_search.best_estimator_

        # Log best params manually
        mlflow.log_params(rf_search.best_params_)
        mlflow.log_metric("rf_best_cv_score", rf_search.best_score_)

        mlflow.log_params({f"et_{k}": v for k, v in et_search.best_params_.items()})
        mlflow.log_metric("et_best_cv_score", et_search.best_score_)

        # Evaluate
        y_pred_rf = best_rf.predict(X_test)
        y_pred_et = best_et.predict(X_test)

        rf_f1 = f1_score(y_test, y_pred_rf)
        et_f1 = f1_score(y_test, y_pred_et)

        mlflow.log_metric("rf_test_f1", rf_f1)
        mlflow.log_metric("et_test_f1", et_f1)

        print("RF Test F1:", rf_f1)
        print("ET Test F1:", et_f1)

        return best_rf, best_et
quick_tune_models(X_train, X_test, y_train, y_test)

You want to extract only the Data Transformation part from your full notebook (remove EDA, modeling, tuning, MLflow, etc.) and keep it clean for a pipeline step.

Below is the clean, production-ready Data Transformation module based on your code.

✅ What Belongs to Data Transformation?
✔ Remove duplicates
✔ Create binary target
✔ Train-test split (stratified)
✔ Log transform skewed columns
✔ Scaling features
✔ Return transformed datasets

❌ NOT included:

Visualization

Model training

Model evaluation

Hyperparameter tuning

MLflow

✅ Clean Data Transformation Script
import os
import pandas as pd
import numpy as np
from dataclasses import dataclass
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transformer_path: Path


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def initiate_data_transformation(self):

        # ===============================
        # 1️⃣ Load Data
        # ===============================
        df = pd.read_csv(self.config.data_path)

        # Remove duplicates
        df.drop_duplicates(inplace=True)

        # ===============================
        # 2️⃣ Create Binary Target
        # ===============================
        df["quality_binary"] = (df["quality"] >= 6).astype(int)

        # ===============================
        # 3️⃣ Define Features & Target
        # ===============================
        X = df.drop(columns=["quality", "quality_binary"])
        y = df["quality_binary"]

        # ===============================
        # 4️⃣ Train-Test Split (Stratified)
        # ===============================
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.30,
            random_state=42,
            stratify=y
        )

        # ===============================
        # 5️⃣ Log Transform Skewed Columns
        # ===============================
        skewed_cols = [
            'residual sugar',
            'chlorides',
            'free sulfur dioxide',
            'total sulfur dioxide',
            'sulphates'
        ]

        X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
        X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

        # ===============================
        # 6️⃣ Feature Scaling
        # ===============================
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Save scaler
        os.makedirs(self.config.root_dir, exist_ok=True)
        joblib.dump(scaler, self.config.transformer_path)

        return (
            X_train_scaled,
            X_test_scaled,
            y_train,
            y_test
        )
✅ What This Gives You
Your transformation stage now:

Cleans data

Creates target

Splits dataset

Handles skewness

Scales features

Saves scaler for inference

Returns processed arrays

This is exactly what belongs inside:

components/data_transformation.py
🚀 Even More Professional (Optional)
Best practice is to use a Pipeline + ColumnTransformer instead of manual log + scaling.

If you want, I can give you the industry-level production version that integrates directly with Model Trainer and works perfectly with MLflow.

If you'd like, next I can help you:

✅ Convert this into full ML project pipeline structure

✅ Add logging

✅ Add artifact saving (train.npy, test.npy)

✅ Make it interview-ready

Just tell me 👌


is this path is okay
data_transformation:
   root_dir: artifacts/data_transformation 
   data_path : artifacts/data_ingestion/Drinks-data.csv
   transformer_path : artifacts/data_transformation

Short answer: ❌ Not fully correct.

Your transformer_path is pointing to a folder, not a file.

Right now you have:

data_transformation:
   root_dir: artifacts/data_transformation 
   data_path: artifacts/data_ingestion/Drinks-data.csv
   transformer_path: artifacts/data_transformation
⚠️ Problem:

transformer_path: artifacts/data_transformation
This is a directory, but in your code you are doing:

joblib.dump(scaler, self.config.transformer_path)
joblib.dump() expects a file path, not a folder.

✅ Correct Version
You should specify a filename:

data_transformation:
   root_dir: artifacts/data_transformation
   data_path: artifacts/data_ingestion/Drinks-data.csv
   transformer_path: artifacts/data_transformation/scaler.pkl
Now:

root_dir → folder

transformer_path → file inside that folder

✔ This is correct structure.

💡 Best Practice (Recommended Structure)
Even better:

data_transformation:
   root_dir: artifacts/data_transformation
   data_path: artifacts/data_ingestion/Drinks-data.csv
   transformer_path: artifacts/data_transformation/preprocessor.pkl
Why preprocessor.pkl?

Because later you may store:

ColumnTransformer

Pipeline

Feature engineering steps

Not just a scaler.

🔎 Final Checklist
Make sure in your code you have:

os.makedirs(self.config.root_dir, exist_ok=True)
joblib.dump(scaler, self.config.transformer_path)
Then everything will work perfectly.

If you'd like, I can also show you the complete professional YAML structure for all pipeline stages (ingestion → validation → transformation → trainer).


why we are storing scaler?

Excellent question 👏 — this is very important in ML engineering.

🎯 Why Do We Store the Scaler?
Because the model must see data in the exact same format during inference as it saw during training.

🚨 What Happens If You Don’t Save It?
During training:

scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
The scaler learns:

mean of each feature

standard deviation of each feature

Now imagine new unseen data comes:

new_data = [...]
model.predict(new_data)
If you don’t apply the same scaler, the feature values will be on a completely different scale.

👉 Result: Wrong predictions

🧠 Example
Assume feature = alcohol %

Training data:

mean = 10
std = 2
Scaler transforms:

(12 - 10) / 2 = 1
But if at prediction time you don’t use the saved scaler:

Model receives:

12 instead of 1
That completely changes model behavior.

🔥 Real-World ML Pipeline
Production flow:

User Input → Load Saved Scaler → Transform → Load Model → Predict
If you don't save the scaler:

You can't transform new data properly

Your model becomes unusable in production

🏢 Industry Practice
In real ML systems we always save:

✅ Preprocessing object (Scaler / ColumnTransformer)

✅ Model

✅ Sometimes encoders

Usually as:

artifacts/
   model.pkl
   preprocessor.pkl
💎 Even More Professional Approach
Instead of saving only the scaler, we save a Pipeline:

Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
Then we only save:

joblib.dump(pipeline, "model.pkl")
And at inference:

model = joblib.load("model.pkl")
model.predict(new_data)
🔥 This is cleaner and industry-grade.

🚀 When Is It Mandatory?
You MUST save preprocessing if you use:

StandardScaler

RobustScaler

MinMaxScaler

OneHotEncoder

LabelEncoder

Log transformation

Feature engineering

PCA

Basically → any transformation learned from data.

🧩 Simple Rule
If you fit() something on training data, you must save it.

If you want, I can now show you:

The correct production inference script

OR the professional full pipeline version (no separate scaler saving needed)

Tell me which level you want 👌


and skewness?

Excellent 👏 — this is the exact next-level thinking you should have.

You asked:

What about skewness? Do we need to save that too?

🎯 Short Answer
✅ YES — if you transform skewed columns, that transformation must also be applied during inference.

But…

❌ You don’t “save skewness”
✅ You save the transformation logic

🔎 In Your Code
You did:

skewed_cols = [
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'sulphates'
]

X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
X_test[skewed_cols] = np.log1p(X_test[skewed_cols])
That means your model was trained on log-transformed values.

If you skip this during prediction:

model.predict(new_data)  # ❌ without log transform
Your model will receive completely different feature distributions.

👉 Predictions will be wrong.

🧠 Important Difference
There are two types of transformations:

1️⃣ Learned transformations (must be saved)
StandardScaler

RobustScaler

PCA

OneHotEncoder

These use .fit() and learn parameters from data.

2️⃣ Fixed transformations (don’t need saving, but must be repeated)
np.log1p()

squaring

manual feature creation

ratio features

They don’t learn anything — but you must apply them again.

🚀 Best Practice (Industry Way)
Instead of manually doing log + scaling, use a Pipeline + ColumnTransformer.

Example:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
import numpy as np

skewed_cols = [
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'sulphates'
]

log_transformer = FunctionTransformer(np.log1p)

preprocessor = ColumnTransformer([
    ("log", log_transformer, skewed_cols),
    ("scale", StandardScaler(), X.columns)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])
Now you only save:

joblib.dump(pipeline, "model.pkl")
🔥 That saves:

log transformation

scaling

model

All in one object.

🏆 Professional Rule
If transformation is:

Type	Save?
Learned from data	✅ MUST save
Fixed mathematical transform	❌ No saving needed
But must reapply at inference	✅ YES
🎯 Best Architecture
Instead of:

log → scale → model
Do:

Pipeline(preprocessing + model)
Save one object.

Cleaner.
Safer.
Industry-grade.

If you want, I can now:

Rewrite your entire transformation using ColumnTransformer properly

Or show you how to modify your current project structure professionally

Which one do you want?


and u did not save train test data also for modling 

Very sharp observation 👏 — you’re thinking like an ML engineer now.

You’re right:

In a proper ML pipeline project, we usually save the transformed train and test datasets as artifacts for the modeling stage.

🎯 Why Should We Save Train/Test After Transformation?
Because in a modular pipeline:

Data Ingestion  →  Data Validation  →  Data Transformation  →  Model Trainer
Each stage should:

Take input

Process it

Save output artifacts

Return paths

So the Model Trainer does NOT redo transformation.

✅ What Should Be Saved?
After transformation you should save:

X_train

X_test

y_train

y_test

preprocessor.pkl (or scaler)

🏢 Industry Folder Structure
artifacts/
    data_transformation/
        X_train.npy
        X_test.npy
        y_train.npy
        y_test.npy
        preprocessor.pkl
✅ Correct Professional Transformation Code
Here is the upgraded version of your transformation class:

import os
import numpy as np
import pandas as pd
import joblib
from dataclasses import dataclass
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transformer_path: Path
    X_train_path: Path
    X_test_path: Path
    y_train_path: Path
    y_test_path: Path


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def initiate_data_transformation(self):

        df = pd.read_csv(self.config.data_path)

        # Remove duplicates
        df.drop_duplicates(inplace=True)

        # Create binary target
        df["quality_binary"] = (df["quality"] >= 6).astype(int)

        X = df.drop(columns=["quality", "quality_binary"])
        y = df["quality_binary"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.30,
            random_state=42,
            stratify=y
        )

        # Log transform skewed columns
        skewed_cols = [
            'residual sugar',
            'chlorides',
            'free sulfur dioxide',
            'total sulfur dioxide',
            'sulphates'
        ]

        X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
        X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

        # Scaling
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Create directory
        os.makedirs(self.config.root_dir, exist_ok=True)

        # Save arrays
        np.save(self.config.X_train_path, X_train_scaled)
        np.save(self.config.X_test_path, X_test_scaled)
        np.save(self.config.y_train_path, y_train.to_numpy())
        np.save(self.config.y_test_path, y_test.to_numpy())

        # Save scaler
        joblib.dump(scaler, self.config.transformer_path)

        return (
            self.config.X_train_path,
            self.config.X_test_path,
            self.config.y_train_path,
            self.config.y_test_path
        )
✅ Your YAML Should Look Like This
data_transformation:
  root_dir: artifacts/data_transformation
  data_path: artifacts/data_ingestion/Drinks-data.csv
  transformer_path: artifacts/data_transformation/preprocessor.pkl
  X_train_path: artifacts/data_transformation/X_train.npy
  X_test_path: artifacts/data_transformation/X_test.npy
  y_train_path: artifacts/data_transformation/y_train.npy
  y_test_path: artifacts/data_transformation/y_test.npy
🚀 Now Model Trainer Will Do
X_train = np.load(config.X_train_path)
y_train = np.load(config.y_train_path)
Clean separation.
Reusable pipeline.
Production-ready structure.

🧠 Important Concept
There are 2 approaches:

Approach 1 (Modular Project — like yours)
✔ Save transformed datasets
✔ Save preprocessor
✔ Model trainer loads them

Approach 2 (Single Pipeline Object)
✔ Don’t save train/test separately
✔ Save full Pipeline(preprocessor + model)

Both are correct — depends on architecture.

If you want, I can now:

Show the correct ModelTrainer class

Or show the fully professional industry version with ColumnTransformer

Tell me which level you want 🚀


we need to save it or just returning it to pipeline is more then needed

This is a very important architectural question 👌
The answer depends on how your project is designed.

Let’s break it down clearly.

🎯 Option 1 — Return Only (No Saving)
return X_train_scaled, X_test_scaled, y_train, y_test
✅ When This Is Enough
You are running everything in one script

You don’t need stage separation

You’re not building a production ML pipeline

You don’t need experiment reproducibility from saved artifacts

This is common in:

Notebooks

Small ML projects

Kaggle work

🎯 Option 2 — Save as Artifacts (Recommended for ML Projects)
np.save("X_train.npy", X_train_scaled)
✅ When You SHOULD Save
If you are building:

A modular ML project structure

A pipeline like:

ingestion → validation → transformation → trainer
MLflow experiment tracking

Production-level system

Reproducible pipelines

CI/CD ML workflows

Then yes — you should save them.

🏢 Why Saving Is Professional
Because each stage should:

Stage	Input	Output
Ingestion	Raw data	Clean CSV
Validation	CSV	Validated CSV
Transformation	CSV	Transformed arrays
Trainer	Transformed arrays	Model
Each stage produces artifacts, not just memory objects.

This allows:

Rerunning only failed stages

Debugging intermediate outputs

Inspecting transformed data

Reproducibility

Parallel execution

🔥 What Industry Actually Does
There are 2 common real-world patterns:

Pattern A — Save Transformed Data (Modular ML Systems)
Used in:

Enterprise ML systems

Airflow pipelines

Kubeflow

MLOps pipelines

✔ Save train/test
✔ Save preprocessor
✔ Save model

Pattern B — Save Only Pipeline Object (Modern & Cleaner)
Instead of saving train/test, you:

pipeline = Pipeline([
    ("preprocessor", ColumnTransformer(...)),
    ("model", RandomForestClassifier())
])
Train once:

pipeline.fit(X_train, y_train)
joblib.dump(pipeline, "model.pkl")
Then you never separately store transformed arrays.

This is cleaner and more modern.

🎯 So What Should YOU Do?
Since you're building a structured ML project with artifacts folders,
I recommend:

✅ Save:
preprocessor.pkl

model.pkl

❌ Not mandatory:
X_train.npy

X_test.npy

Unless your instructor specifically wants stage outputs saved.

🧠 My Professional Recommendation For You
Instead of saving train/test arrays:

Save only preprocessor.pkl

In ModelTrainer:

Load raw train data

Apply preprocessor

Train model

Save final pipeline

This avoids redundant storage.

💎 Golden Rule
If your pipeline runs end-to-end automatically:

👉 Returning data is enough.

If your stages are independent modules:

👉 Save artifacts.

If you tell me:

Is this for coursework?

Is this for MLOps production style?

Is your trainer stage separate?

I’ll tell you the exact best architecture for your case.


give code to save it
and add looging
#component

import os
import pandas as pd
import numpy as np
from dataclasses import dataclass
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transformer_path: Path


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def initiate_data_transformation(self):

        # ===============================
        # 1️⃣ Load Data
        # ===============================
        df = pd.read_csv(self.config.data_path)

        # Remove duplicates
        df.drop_duplicates(inplace=True)

        # ===============================
        # 2️⃣ Create Binary Target
        # ===============================
        df["quality_binary"] = (df["quality"] >= 6).astype(int)

        # ===============================
        # 3️⃣ Define Features & Target
        # ===============================
        X = df.drop(columns=["quality", "quality_binary"])
        y = df["quality_binary"]

        # ===============================
        # 4️⃣ Train-Test Split (Stratified)
        # ===============================
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.30,
            random_state=42,
            stratify=y
        )

        # ===============================
        # 5️⃣ Log Transform Skewed Columns
        # ===============================
        skewed_cols = [
            'residual sugar',
            'chlorides',
            'free sulfur dioxide',
            'total sulfur dioxide',
            'sulphates'
        ]

        X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
        X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

        # ===============================
        # 6️⃣ Feature Scaling
        # ===============================
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Save scaler
        os.makedirs(self.config.root_dir, exist_ok=True)
        joblib.dump(scaler, self.config.transformer_path)

        return (
            X_train_scaled,
            X_test_scaled,
            y_train,
            y_test
        )

Perfect 👍 I’ll upgrade your component properly:

We will add:

✅ Logging

✅ Saving transformed train/test arrays

✅ Saving scaler

✅ Clean artifact structure

✅ Proper directory handling

I’ll assume you already have:

from mlProject import logger
If not, tell me and I’ll give you a logger setup file too.

✅ Updated DataTransformation Component (With Saving + Logging)
import os
import pandas as pd
import numpy as np
from dataclasses import dataclass
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
from mlProject import logger


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transformer_path: Path
    X_train_path: Path
    X_test_path: Path
    y_train_path: Path
    y_test_path: Path


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def initiate_data_transformation(self):

        try:
            logger.info("Starting Data Transformation")

            # ===============================
            # 1️⃣ Load Data
            # ===============================
            logger.info("Loading dataset")
            df = pd.read_csv(self.config.data_path)
            logger.info(f"Dataset loaded with shape: {df.shape}")

            # Remove duplicates
            df.drop_duplicates(inplace=True)
            logger.info(f"Shape after removing duplicates: {df.shape}")

            # ===============================
            # 2️⃣ Create Binary Target
            # ===============================
            logger.info("Creating binary target column")
            df["quality_binary"] = (df["quality"] >= 6).astype(int)

            # ===============================
            # 3️⃣ Define Features & Target
            # ===============================
            X = df.drop(columns=["quality", "quality_binary"])
            y = df["quality_binary"]

            # ===============================
            # 4️⃣ Train-Test Split
            # ===============================
            logger.info("Splitting data into train and test")
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.30,
                random_state=42,
                stratify=y
            )

            logger.info(f"X_train shape: {X_train.shape}")
            logger.info(f"X_test shape: {X_test.shape}")

            # ===============================
            # 5️⃣ Log Transform Skewed Columns
            # ===============================
            logger.info("Applying log transformation to skewed columns")

            skewed_cols = [
                'residual sugar',
                'chlorides',
                'free sulfur dioxide',
                'total sulfur dioxide',
                'sulphates'
            ]

            X_train[skewed_cols] = np.log1p(X_train[skewed_cols])
            X_test[skewed_cols] = np.log1p(X_test[skewed_cols])

            # ===============================
            # 6️⃣ Feature Scaling
            # ===============================
            logger.info("Applying Standard Scaling")

            scaler = StandardScaler()

            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # ===============================
            # 7️⃣ Save Artifacts
            # ===============================
            logger.info("Saving transformation artifacts")

            os.makedirs(self.config.root_dir, exist_ok=True)

            # Save numpy arrays
            np.save(self.config.X_train_path, X_train_scaled)
            np.save(self.config.X_test_path, X_test_scaled)
            np.save(self.config.y_train_path, y_train.to_numpy())
            np.save(self.config.y_test_path, y_test.to_numpy())

            logger.info("Train-test arrays saved successfully")

            # Save scaler
            joblib.dump(scaler, self.config.transformer_path)
            logger.info("Scaler saved successfully")

            logger.info("Data Transformation Completed Successfully")

            return (
                self.config.X_train_path,
                self.config.X_test_path,
                self.config.y_train_path,
                self.config.y_test_path
            )

        except Exception as e:
            logger.exception("Error occurred during Data Transformation")
            raise e
✅ Your YAML Should Now Be
data_transformation:
  root_dir: artifacts/data_transformation
  data_path: artifacts/data_ingestion/Drinks-data.csv
  transformer_path: artifacts/data_transformation/scaler.pkl
  X_train_path: artifacts/data_transformation/X_train.npy
  X_test_path: artifacts/data_transformation/X_test.npy
  y_train_path: artifacts/data_transformation/y_train.npy
  y_test_path: artifacts/data_transformation/y_test.npy
🔥 What This Gives You
Now your transformation stage:

✔ Logs every step
✔ Saves scaled train/test
✔ Saves scaler
✔ Returns artifact paths
✔ Follows modular ML architecture

This is now project-level clean code.

If you want next, I can give you:

✅ ModelTrainer component

✅ Logger configuration file

✅ Fully industry-level Pipeline version (cleaner than this)

✅ How to load these artifacts inside trainer

Tell me the next step 🚀


any change in config manager
#configManger

class ConfigrationManger:
    def __init__(
        self,
        config_filepath  = CONFIG_FILE_PATH ,
        params_filepath = PARAMS_FILE_PATH ,
        schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath) 
        
        create_directories(self.config.root_dir) 
        
        
        
    def get_data_transformation_config(self)->DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directories([config.root_dir])
        
        
        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            
            data_path = config.data_path 
        )
        
        return data_transformation_config

Yes ✅ — you MUST update your ConfigurationManager.

Right now your DataTransformationConfig expects:

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    transformer_path: Path
    X_train_path: Path
    X_test_path: Path
    y_train_path: Path
    y_test_path: Path
But your ConfigurationManager is only passing:

root_dir
data_path
So yes — changes are required.

✅ Corrected ConfigurationManager
Here is the fixed and clean version:

from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories
from mlProject.entity import DataTransformationConfig


class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            transformer_path=config.transformer_path,
            X_train_path=config.X_train_path,
            X_test_path=config.X_test_path,
            y_train_path=config.y_train_path,
            y_test_path=config.y_test_path,
        )

        return data_transformation_config
🔥 Important Fixes
1️⃣ Fix this line in your original code:
❌ Wrong:

create_directories(self.config.root_dir)
✔ Correct:

create_directories([self.config.artifacts_root])
Because create_directories() expects a list.

✅ Make Sure YAML Matches
Your config.yaml must contain:

artifacts_root: artifacts

data_transformation:
  root_dir: artifacts/data_transformation
  data_path: artifacts/data_ingestion/Drinks-data.csv
  transformer_path: artifacts/data_transformation/scaler.pkl
  X_train_path: artifacts/data_transformation/X_train.npy
  X_test_path: artifacts/data_transformation/X_test.npy
  y_train_path: artifacts/data_transformation/y_train.npy
  y_test_path: artifacts/data_transformation/y_test.npy
🚀 Now Flow Will Be
ConfigurationManager
        ↓
DataTransformationConfig
        ↓
DataTransformation Component
        ↓
Saved Artifacts
Now your pipeline structure is correct.

If you want next step, I can give you:

✅ ModelTrainer configuration + component

✅ Complete project flow script (main.py)

✅ How to load these saved numpy files inside trainer

Tell me what you want next 🚀


give this type of block to test
try: 
    config= ConfigurationManager()
    data_validation_config  = config.get_data_validation_config()
    data_validation = DataValidation(data_validation_config)
    data_validation.validate_all_columns()
    
except Exception as e:
    raise e

Here is the same type of testing block but for your Data Transformation stage ✅

✅ Test Block for Data Transformation
try:
    config = ConfigurationManager()
    
    data_transformation_config = config.get_data_transformation_config()
    
    data_transformation = DataTransformation(config=data_transformation_config)
    
    X_train_path, X_test_path, y_train_path, y_test_path = \
        data_transformation.initiate_data_transformation()
    
    print("Data Transformation Completed Successfully!")
    print("Saved Files:")
    print("X_train:", X_train_path)
    print("X_test :", X_test_path)
    print("y_train:", y_train_path)
    print("y_test :", y_test_path)

except Exception as e:
    raise e
✅ If You Want to Verify Saved Files
After running, you can also quickly check:

import numpy as np

X_train = np.load(X_train_path)
y_train = np.load(y_train_path)

print("Loaded X_train shape:", X_train.shape)
print("Loaded y_train shape:", y_train.shape)
🔥 What This Does
This block:

Initializes ConfigurationManager

Gets DataTransformationConfig

Runs transformation

Saves:

X_train.npy

X_test.npy

y_train.npy

y_test.npy

scaler.pkl

Prints artifact paths

Now your testing structure is consistent with:

Data Ingestion
Data Validation
Data Transformation
If you want next, I can give you:

✅ Model Trainer test block

✅ Full pipeline run script (main.py)

✅ Complete project execution order

Tell me the next stage 🚀



